In [1]:
!pip install scholarly habanero sentence-transformers requests beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 20.7 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.4-py3-none-any.whl size=43609 sha256=acec80bdbdaa69c97b2f14f1bee816e7a3909071be2d675859caf86a74a70706
  Stored in directory: /root/.cache/pip/wheels/54/f8/e6/ecfceb6af875ddc5096bb3811795ac336f50371009a601454d
  Created wheel for free-proxy: filename=free_proxy-1.1.3-py3-none-any.whl size=6097 sha256=f6

In [ ]:
import os
import time
import random
import re
import requests
import pandas as pd
import warnings
from datetime import datetime
from habanero import Crossref
from sentence_transformers import SentenceTransformer, util
from google.colab import drive

# 1. SETUP & PATHS
warnings.filterwarnings("ignore")
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/uMich/Research/Cernak Lab - Trees/PhD-ish/Data/3 24/SystematicReview_Project"
DOWNLOAD_DIR = os.path.join(BASE_PATH, "PDFs")
LOG_FILE = os.path.join(BASE_PATH, f"download_log_{datetime.now().strftime('%H%M')}.csv")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("🚀 Initializing Crossref-to-PDF Engine...")
vector_model = SentenceTransformer('all-MiniLM-L6-v2')
cr = Crossref()
print(f"✅ Tools Ready. Saving to: {DOWNLOAD_DIR}")

# 2. UTILITIES
def heartbeat(msg):
    print(f"  [{datetime.now().strftime('%H:%M:%S')}] {msg}")

def staggered_wait(label="Wait", max_s=3):
    t = random.uniform(1.0, max_s)
    heartbeat(f"{label}: {t:.1f}s...")
    time.sleep(t)

def clean_filename(title):
    name = re.sub(r'[\\/*?:"<>|]', '', title).strip().replace(' ', '_')
    return name[:80] + ".pdf"

def extract_title(ref):
    """Extracts title from quotes to drastically improve Semantic Match accuracy."""
    quoted = re.findall(r'["\'](.*?)["\']', ref)
    if quoted: return quoted[0]
    clean = re.sub(r'^\d+[\.\)]\s*', '', ref)
    return clean.strip()

# 3. ADVANCED PDF FETCHING (Fixes the corrupt PDF issue)
def verify_and_save_pdf(url, filepath):
    """Downloads fully into RAM, verifies PDF header, and saves cleanly."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
        'Accept': 'application/pdf,application/xhtml+xml,text/html;q=0.9,*/*;q=0.8'
    }
    try:
        r = requests.get(url, timeout=25, headers=headers)
        if r.status_code == 200:
            pdf_bytes = r.content
            # The magic fix: Check if '%PDF' exists in the first 100 bytes
            if b'%PDF' in pdf_bytes[:100]:
                with open(filepath, 'wb') as f:
                    f.write(pdf_bytes)
                return "Success"
            else:
                return "Failed Byte Check (HTML/Login Page)"
        return f"HTTP {r.status_code}"
    except Exception as e:
        return f"DL Error: {str(e)[:20]}"

# 4. MAIN WORKFLOW
def run_hybrid_workflow(reference_text):
    refs = [r.strip() for r in reference_text.strip().split('\n') if r.strip()]
    history = []

    for i, raw_ref in enumerate(refs):
        print(f"\n📄 ITEM {i+1}/{len(refs)}: {raw_ref[:70]}...")

        search_target = extract_title(raw_ref)
        status, source, final_url, doi = "Failed", "None", None, "N/A"

        try:
            # Step 1: Use Crossref to find the exact DOI based on the citation
            res = cr.works(query=raw_ref, limit=1)

            if res['message']['items']:
                item = res['message']['items'][0]
                doi = item.get('DOI', 'N/A')
                found_title = item.get('title', ['Unknown'])[0]

                # Semantic match against the extracted title to avoid downloading reviews
                emb1 = vector_model.encode(search_target, convert_to_tensor=True)
                emb2 = vector_model.encode(found_title, convert_to_tensor=True)
                score = float(util.cos_sim(emb1, emb2))
                heartbeat(f"Match: {score:.2f} | Found Title: {found_title[:45]}...")

                if score > 0.55:

                    # Step 2: Look for Open Access links using the exact DOI
                    if doi != 'N/A':
                        # Try Unpaywall First
                        heartbeat("Checking Unpaywall via DOI...")
                        upw_url = f"https://api.unpaywall.org/v2/{doi}?email=researcher@umich.edu"
                        try:
                            r_upw = requests.get(upw_url, timeout=10)
                            if r_upw.status_code == 200 and r_upw.json().get('is_oa'):
                                # Safely check for best OA location
                                best_oa = r_upw.json().get('best_oa_location')
                                if best_oa:
                                    final_url = best_oa.get('url_for_pdf')
                                    if final_url: source = "Unpaywall"
                        except: pass

                        # Fallback to Semantic Scholar if Unpaywall is empty
                        if not final_url:
                            heartbeat("Checking Semantic Scholar via DOI...")
                            ss_url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=openAccessPdf"
                            try:
                                r_ss = requests.get(ss_url, timeout=10)
                                if r_ss.status_code == 200:
                                    oa_data = r_ss.json().get('openAccessPdf')
                                    if oa_data and oa_data.get('url'):
                                        final_url = oa_data['url']
                                        source = "Semantic Scholar"
                            except: pass

                    # Step 3: Download & Verify the PDF
                    if final_url:
                        heartbeat(f"Attempting Download from {source}...")
                        file_name = clean_filename(found_title)
                        save_path = os.path.join(DOWNLOAD_DIR, file_name)

                        dl_result = verify_and_save_pdf(final_url, save_path)

                        if dl_result == "Success":
                            status = "Success"
                            heartbeat(f"✅ SAVED CLEAN PDF: {file_name}")
                        else:
                            status = dl_result
                            heartbeat(f"❌ {dl_result}")
                    else:
                        status = "No OA Link Found"
                        heartbeat("❌ Paper is Paywalled. No Open Access links available.")
                else:
                    status = "Semantic Mismatch"
                    heartbeat("⚠️ Match score too low (Likely a review/different paper).")
            else:
                status = "Not Found in Crossref"
                heartbeat("❌ Crossref could not resolve this citation.")

        except Exception as e:
            status = f"Crash: {str(e)[:20]}"
            heartbeat(f"💥 Error: {status}")

        history.append({"Ref": raw_ref[:50], "Status": status, "Source": source, "DOI": doi})
        staggered_wait("Inter-paper Cooldown", 3)

    pd.DataFrame(history).to_csv(LOG_FILE, index=False)
    print("\n" + "="*50)
    print(f"Done! Check your Drive folder: {DOWNLOAD_DIR}")
    print("="*50)

# 5. RUN WITH REAL OPEN ACCESS PAPERS
real_oa_refs = """

"""

run_hybrid_workflow(real_oa_refs)

Mounted at /content/drive
🚀 Initializing Crossref-to-PDF Engine...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Tools Ready. Saving to: /content/drive/MyDrive/uMich/Research/Cernak Lab - Trees/PhD-ish/Data/3 24/SystematicReview_Project/PDFs

📄 ITEM 1/3: 1. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gom...
  [21:44:19] Match: 0.98 | Found Title: Attention Is All You Need...
  [21:44:19] Checking Unpaywall via DOI...
  [21:44:19] Attempting Download from Unpaywall...
  [21:44:21] ❌ HTTP 404
  [21:44:21] Inter-paper Cooldown: 1.2s...

📄 ITEM 2/3: 2. Jumper, J., Evans, R., Pritzel, A., Green, T., Figurnov, M., Ronneb...
  [21:44:23] Match: 0.99 | Found Title: Highly accurate protein structure prediction ...
  [21:44:23] Checking Unpaywall via DOI...
  [21:44:23] Attempting Download from Unpaywall...
  [21:44:24] ✅ SAVED CLEAN PDF: Highly_accurate_protein_structure_prediction_with_AlphaFold.pdf
  [21:44:24] Inter-paper Cooldown: 1.4s...

📄 ITEM 3/3: 3. Bollen, J., Van de Sompel, H., Hagberg, A., & Chute, R. (2009). "A ...
  [21:44:26] Match: 0.99 | Found Title: A Principal Comp

In [1]:
target_references = """
■ REFERENCES
(1) Mahjour, B.; Shen, Y.; Cernak, T. Ultrahigh-throughput
experimentation for information-rich chemical synthesis. Acc. Chem.
Res. 2021, 54, 2337−2346.
(2) Gesmundo, N. J.; Sauvagnat, B.; Curran, P. J.; Richards, M. P.;
Andrews, C. L.; Dandliker, P. J.; Cernak, T. Nanoscale synthesis and
affinity ranking. Nature 2018, 557, 228−232.
(3) Uehling, M. R.; King, R. P.; Krska, S. W.; Cernak, T.; Buchwald,
S. L. Pharmaceutical diversification via palladium oxidative addition
complexes. Science 2019, 363, 405.
(4) Lin, S.; Dikler, S.; Blincoe, W. D.; Ferguson, R. D.; Sheridan, R.
P.; Peng, Z.; Conway, D. V.; Zawatzky, K.; Wang, H.; Cernak, T.;
Davies, I. W.; DiRocco, D. A.; Sheng, H.; Welch, C. J.; Dreher, S. D.
Mapping the dark space of chemical reactions with extended
nanomole synthesis and MALDI-TOF MS. Science 2018, 361,
No. eaar6236.
(5) Buitrago Santanilla, A.; Regalado, E. L.; Pereira, T.; Shevlin, M.;
Bateman, K.; Campeau, L.-C.; Schneeweis, J.; Berritt, S.; Shi, Z.-C.;
Nantermet, P.; Liu, Y.; Helmy, R.; Welch, C. J.; Vachal, P.; Davies, I.
W.; Cernak, T.; Dreher, S. D. Nanomole-scale high-throughput
chemistry for the synthesis of complex molecules. Science 2015, 347,
49.
(6) Shen, Y.; Borowski, J. E.; Hardy, M. A.; Sarpong, R.; Doyle, A.
G.; Cernak, T. Automation and computer-assisted planning for
chemical synthesis. Nat. Rev. Methods Primers 2021, 1, 23.
(7) Torres, J. A. G.; Lau, S. H.; Anchuri, P.; Stevens, J. M.; Tabora, J.
E.; Li, J.; Borovika, A.; Adams, R. P.; Doyle, A. G. A Multi-Objective
Active Learning Platform and Web App for Reaction Optimization. J.
Am. Chem. Soc. 2022, 144, 19999−20007.
(8) Shields, B. J.; Stevens, J.; Li, J.; Parasram, M.; Damani, F.;
Alvarado, J. I. M.; Janey, J. M.; Adams, R. P.; Doyle, A. G. Bayesian
reaction optimization as a tool for chemical synthesis. Nature 2021,
590, 89−96.
(9) Williams, W. L.; Zeng, L.; Gensch, T.; Sigman, M. S.; Doyle, A.
G.; Anslyn, E. V. The evolution of data-driven modeling in organic
chemistry. ACS Cent. Sci. 2021, 7, 1622−1637.
(10) Gao, W.; Raghavan, P.; Coley, C. W. Autonomous platforms for
data-driven organic synthesis. Nat. Commun. 2022, 13, 1075.
(11) Tu, Z.; Stuyver, T.; Coley, C. W. Predictive chemistry: Machine
learning for reaction deployment, reaction development, and reaction
discovery. Chem. Sci. 2023, 14, 226−244.
(12) Shim, E.; Kammeraad, J. A.; Xu, Z.; Tewari, A.; Cernak, T.;
Zimmerman, P. M. Predicting reaction conditions from limited data
through active transfer learning. Chem. Sci. 2022, 13, 6655−6668.
(13) Luo, R.; Sun, L.; Xia, Y.; Qin, T.; Zhang, S.; Poon, H.; Liu, T.-
Y. BioGPT: generative pre-trained transformer for biomedical text
generation and mining. Briefings Bioinformat. 2022, 23, No. bbac409.
(14) Ferruz, N.; Schmidt, S.; Höcker, B. ProtGPT2 is a deep
unsupervised language model for protein design. Nat. Commun. 2022,
13, 4348.
(15) Mahjour, B.; Zhang, R.; Shen, Y.; McGrath, A.; Zhao, R.;
Mohamed, O. G.; Lin, Y.; Zhang, Z.; Douthwaite, J. L.; Tripathi, A.;
Cernak, T. Rapid Planning and Analysis of High-Throughput
Experiment Arrays for Reaction Discovery. Nat. Commun. 2023, 14,
3924.
(16) Bostrom, J.; Brown, D. G.; Young, R. J.; Keseru, G. M.
Expanding the medicinal chemistry synthetic toolbox. Nat. Rev. Drug
Discovery 2018, 17, 709−727.
(17) OpenAI. https://openai.com/ (accessed February 2023).
(18) Zhang, C.; Trudell, M. L. Palladium-bisimidazol-2-ylidene
complexes as catalysts for general and efficient Suzuki cross-coupling
reactions of aryl chlorides with arylboronic acids. Tetrahedron Lett.
2000, 41, 595−598.
(19) Choy, P. Y.; Yuen, O. Y.; Leung, M. P.; Chow, W. K.; Kwong,
F. Y. A Highly Efficient Monophosphine Ligand for Parts per Million
Levels Pd-Catalyzed Suzuki−Miyaura Coupling of (Hetero) Aryl
Chlorides. Eur. J. Org. Chem. 2020, 2020, 2846−2853.
(20) Thomas, A. A.; Denmark, S. E. Pre-transmetalation
intermediates in the Suzuki-Miyaura reaction revealed: The missing
link. Science 2016, 352, 329−332.
(21) Zhao, X.-Y.; Zhou, Q.; Lu, J.-M. Synthesis and characterization
of N-heterocyclic carbene-palladium (ii) chlorides-1-methylindazole
and-1-methylpyrazole complexes and their catalytic activity toward
C−N coupling of aryl chlorides. RSC Adv. 2016, 6, 24484−24490.
(22) King, A. K.; Brar, A.; Findlater, M. A tertiary phosphine oxide
ligand-based recyclable system for the Suzuki−Miyaura and Negishi
reactions: evidence for pseudo-homogeneous catalysis. Catal. Sci.
Technol. 2023, 13, 301−304.
(23) Prima, D. O.; Madiyeva, M.; Burykina, J. V.; Minyaev, M. E.;
Boiko, D. A.; Ananikov, V. P. Evidence for “cocktail”-type catalysis in
Buchwald−Hartwig reaction. A mechanistic study. Catal. Sci. Technol.
2021, 11, 7171−7188.
(24) Ondar, E. E.; Burykina, J. V.; Ananikov, V. P. Evidence for the
“cocktail” nature of platinum-catalyzed alkyne and alkene hydrosilylation
reactions. Catal. Sci. Technol. 2022, 12, 1173−1186.
(25) Fors, B. P.; Buchwald, S. L. A multiligand based Pd catalyst for
C− N cross-coupling reactions. J. Am. Chem. Soc. 2010, 132, 15914−
15917.
(26) NIH. https://pubchem.ncbi.nlm.nih.gov/ (accessed February
2023).
Organic Process Research & Development pubs.acs.org/OPRD Article
https
"""

In [4]:
# ==========================================
# 0. AUTO-INSTALL DEPENDENCIES
# ==========================================
import os
print("📦 Installing required packages (curl_cffi, bs4, habanero, sentence-transformers)...")
os.system('pip install -q curl_cffi beautifulsoup4 habanero sentence-transformers pandas urllib3')

import time
import random
import re
import difflib
import pandas as pd
import warnings
import urllib3
from datetime import datetime
from habanero import Crossref
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests
from sentence_transformers import SentenceTransformer, util
from google.colab import drive
import requests
from IPython.display import display, HTML

# Disable SSL warnings for our fallback net
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ==========================================
# 1. API KEYS & CREDENTIALS
# ==========================================
YOUR_EMAIL = "your_email@umich.edu"
OPENALEX_API_KEY = ""
SEMANTIC_SCHOLAR_API_KEY = ""
CORE_API_KEY = ""

# ==========================================
# 2. SETUP & PATHS
# ==========================================
warnings.filterwarnings("ignore")
drive.mount('/content/drive', force_remount=True)

BASE_PATH = "/content/drive/MyDrive/uMich/Research/Cernak Lab - Trees/PhD-ish/Data/3 24/SystematicReview_Project"
DOWNLOAD_DIR = os.path.join(BASE_PATH, "PDFs")
LOG_FILE = os.path.join(BASE_PATH, f"verbose_log_{datetime.now().strftime('%H%M')}.csv")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("🚀 Initializing APEX Engine (Full Automation + Manual Fallback Dashboard)...")
vector_model = SentenceTransformer('all-MiniLM-L6-v2')
cr = Crossref(mailto=YOUR_EMAIL)
print(f"✅ Tools Ready. Saving PDFs to: {DOWNLOAD_DIR}")

# ==========================================
# 3. UTILITIES & SMART PARSING
# ==========================================
def heartbeat(msg):
    print(f"  [{datetime.now().strftime('%H:%M:%S')}] {msg}")

def staggered_wait(label="Wait", max_s=3):
    t = random.uniform(1.0, max_s)
    heartbeat(f"{label}: {t:.1f}s...")
    time.sleep(t)

def clean_filename(title):
    name = re.sub(r'[\\/*?:"<>|]', '', title).strip().replace(' ', '_')
    return name[:80] + ".pdf"

def parse_references(text):
    """Intelligently stitches multi-line PDF copy-pastes back into single citations."""
    text = text.replace('−', '-').replace('—', '-')
    lines = text.split('\n')
    refs = []
    current_ref = ""
    start_pattern = re.compile(r'^\s*\(?\d+[a-z]?[\.\)]\s+')

    for line in lines:
        line = line.strip()
        if not line: continue
        if "■ REFERENCES" in line or line.startswith("Organic Process Research") or line == "https":
            continue

        if start_pattern.match(line):
            if current_ref: refs.append(current_ref.strip())
            current_ref = line
        else:
            if current_ref:
                if current_ref.endswith('-'): current_ref = current_ref[:-1] + line
                else: current_ref += " " + line
            else: current_ref = line

    if current_ref: refs.append(current_ref.strip())
    return refs

def extract_title(ref):
    quoted = re.findall(r'["\'](.*?)["\']', ref)
    if quoted: return quoted[0]
    clean = re.sub(r'^\s*\(?\d+[a-z]?[\.\)]\s*', '', ref)
    return clean.strip()

def extract_doi_regex(ref):
    match = re.search(r'(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', ref, re.I)
    if match: return match.group(1).rstrip('.')
    return None

def fuzzy_match(s1, s2):
    return difflib.SequenceMatcher(None, s1.lower(), s2.lower()).ratio()

def rewrite_direct_pdf_url(url):
    if not url: return url
    if url.startswith('http://'): url = url.replace('http://', 'https://')

    if 'arxiv.org/abs/' in url: url = url.replace('arxiv.org/abs/', 'arxiv.org/pdf/')
    elif 'nature.com/articles/' in url: url = url.replace('/articles/', '/pdf/')
    elif 'link.springer.com/article/' in url: url = url.replace('/article/', '/content/pdf/')
    elif 'onlinelibrary.wiley.com/doi/' in url and '/pdf/' not in url: url = url.replace('/doi/', '/doi/pdfdirect/')

    if not url.lower().endswith('.pdf') and 'arxiv.org' not in url and 'wiley.com' not in url:
        url += '.pdf'

    return url.replace('.pdf.pdf', '.pdf')

def get_base_url(url):
    parsed_uri = urlparse(url)
    return '{uri.scheme}://{uri.netloc}/'.format(uri=parsed_uri)

def generate_umich_proxy(doi):
    if not doi or doi == 'N/A': return ""
    return f"https://proxy.lib.umich.edu/login?url=https://doi.org/{doi}"

# ==========================================
# 4. ADVANCED SPOOFING & HTML SCRAPING
# ==========================================
def check_wayback_machine(url):
    try:
        r = requests.get(f"http://archive.org/wayback/available?url={url}", timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get("archived_snapshots", {}).get("closest", {}).get("available"):
                return data["archived_snapshots"]["closest"]["url"]
    except: pass
    return None

def get_spoofed_headers(url):
    referers = [
        'https://scholar.google.com/',
        'https://pubmed.ncbi.nlm.nih.gov/',
        'https://www.webofscience.com/',
        'https://t.co/'
    ]
    return {
        'Accept': 'application/pdf,application/xhtml+xml,text/html;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': random.choice(referers),
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

def verify_and_save_pdf(url, filepath, is_retry=False):
    url = rewrite_direct_pdf_url(url)
    headers = get_spoofed_headers(url)
    impersonations = ["chrome110", "edge99", "safari15_3"]
    chosen_browser = random.choice(impersonations)

    content_bytes, status_code = None, None

    # 1. TLS Spoofing
    try:
        r = cffi_requests.get(url, headers=headers, impersonate=chosen_browser, timeout=30, allow_redirects=True, verify=False)
        content_bytes = r.content
        status_code = r.status_code
    except Exception as e:
        heartbeat(f"TLS Spoofing dropped ({str(e)[:15]}). Deploying requests fallback...")
        time.sleep(2)
        # 2. Exponential Backoff Fallback Net
        try:
            r2 = requests.get(url, headers=headers, timeout=30, allow_redirects=True, verify=False)
            content_bytes = r2.content
            status_code = r2.status_code
        except Exception as e2:
            return f"DL Error: Both connection methods failed"

    # 3. Wayback Machine Resurrection Protocol
    if status_code == 404 and not is_retry:
        heartbeat("Hit 404 Dead Link. Asking the Wayback Machine for a backup...")
        archive_url = check_wayback_machine(url)
        if archive_url:
            heartbeat(f"Archived PDF found! Retrying via Wayback...")
            return verify_and_save_pdf(archive_url, filepath, is_retry=True)

    # Analyze payload
    if status_code == 200 and content_bytes:
        # Perfect Match
        if b'%PDF' in content_bytes[:100]:
            with open(filepath, 'wb') as f:
                f.write(content_bytes)
            return "Success"

        # HTML Wall - Deep Dive Scrape
        if not is_retry and b'<html' in content_bytes[:500].lower():
            heartbeat("Hit HTML wall. Deploying Deep Scrape...")
            soup = BeautifulSoup(content_bytes, 'html.parser')

            meta_tags = ['citation_pdf_url', 'wkhealth_pdf_url', 'eprints.pdf_url', 'bepress_citation_pdf_url']
            meta_pdf = soup.find('meta', attrs={'name': meta_tags})

            hidden_url = None
            if meta_pdf and meta_pdf.get('content'):
                hidden_url = meta_pdf['content']
            else:
                pdf_link = soup.find('a', href=re.compile(r'\.pdf$', re.I))
                if pdf_link: hidden_url = pdf_link.get('href')

            if hidden_url:
                hidden_url = urljoin(url, hidden_url)
                heartbeat(f"Found hidden PDF link: {hidden_url[:40]}... Retrying.")
                return verify_and_save_pdf(hidden_url, filepath, is_retry=True)

            return "Failed Byte Check (HTML Wall - No hidden PDF found)"

        return "Failed Byte Check (Unrecognized File Type)"

    return f"HTTP {status_code}"

# ==========================================
# 5. SECURE API FETCHERS (8 APIs)
# ==========================================
def fetch_unpaywall(doi):
    urls = []
    try:
        r = requests.get(f"https://api.unpaywall.org/v2/{doi}?email={YOUR_EMAIL}", timeout=10)
        if r.status_code == 429: time.sleep(4); r = requests.get(f"https://api.unpaywall.org/v2/{doi}?email={YOUR_EMAIL}", timeout=10)
        if r.status_code == 200 and r.json().get('is_oa'):
            data = r.json()
            best = data.get('best_oa_location', {}).get('url_for_pdf')
            if best: urls.append(("Unpaywall (Best)", fix_url(best)))
            for loc in data.get('oa_locations', []):
                alt_url = loc.get('url_for_pdf')
                if alt_url and alt_url != best: urls.append(("Unpaywall (Alt)", fix_url(alt_url)))
    except: pass
    return urls

def fetch_semantic_scholar(doi):
    urls = []
    headers = {'x-api-key': SEMANTIC_SCHOLAR_API_KEY} if SEMANTIC_SCHOLAR_API_KEY else {}
    try:
        r = requests.get(f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=openAccessPdf", headers=headers, timeout=10)
        if r.status_code == 429: time.sleep(4); r = requests.get(f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=openAccessPdf", headers=headers, timeout=10)
        if r.status_code == 200:
            oa_data = r.json().get('openAccessPdf')
            if oa_data and oa_data.get('url'): urls.append(("Semantic Scholar", fix_url(oa_data['url'])))
    except: pass
    return urls

def fetch_openalex(doi):
    urls = []
    url = f"https://api.openalex.org/works/https://doi.org/{doi}?mailto={YOUR_EMAIL}"
    if OPENALEX_API_KEY: url += f"&api_key={OPENALEX_API_KEY}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 429: time.sleep(4); r = requests.get(url, timeout=10)
        if r.status_code == 200:
            best_oa = r.json().get('best_oa_location')
            if best_oa and best_oa.get('pdf_url'): urls.append(("OpenAlex", fix_url(best_oa['pdf_url'])))
    except: pass
    return urls

def fetch_europe_pmc(doi):
    urls = []
    try:
        r = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query=DOI:{doi}&format=json&resultType=core", timeout=10)
        if r.status_code == 200:
            results = r.json().get('resultList', {}).get('result', [])
            if results and results[0].get('pmcid') and results[0].get('isOpenAccess') == 'Y':
                urls.append(("Europe PMC", f"https://europepmc.org/articles/{results[0]['pmcid']}?pdf=render"))
    except: pass
    return urls

def fetch_ncbi_pmc(doi):
    urls = []
    try:
        r = requests.get(f"https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/?ids={doi}&format=json", timeout=10)
        if r.status_code == 200:
            records = r.json().get('records', [])
            if records and records[0].get('pmcid'):
                urls.append(("NCBI PMC", f"https://www.ncbi.nlm.nih.gov/pmc/articles/{records[0]['pmcid']}/pdf/"))
    except: pass
    return urls

def fetch_oaworks(doi):
    urls = []
    try:
        r = requests.get(f"https://api.oa.works/v1/works?id={doi}", timeout=10)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, list) and len(data) > 0: data = data[0]
            url = data.get('pdf_url') or data.get('url')
            if url and '.pdf' in url.lower(): urls.append(("OA.works", fix_url(url)))
    except: pass
    return urls

def fetch_core_ac_uk(doi):
    urls = []
    try:
        r = requests.get(f"https://api.core.ac.uk/v3/discover?doi={doi}", timeout=10)
        if r.status_code == 200:
            data = r.json()
            if data.get('fullTextLink'): urls.append(("CORE.ac.uk", fix_url(data['fullTextLink'])))
    except: pass
    return urls

def fetch_scihub(doi):
    urls = []
    domains = ['https://sci-hub.se/', 'https://sci-hub.st/', 'https://sci-hub.ru/']
    for domain in domains:
        try:
            r = requests.get(f"{domain}{doi}", timeout=10, verify=False)
            if r.status_code == 200:
                soup = BeautifulSoup(r.text, 'html.parser')
                iframe = soup.find('iframe', id='pdf')
                if iframe and iframe.get('src'):
                    src = iframe.get('src')
                    if src.startswith('//'): src = 'https:' + src
                    urls.append(("Sci-Hub", fix_url(src)))
                    break
        except: continue
    return urls

# ==========================================
# 6. THE HYBRID WORKFLOW
# ==========================================
def run_hybrid_workflow(reference_text):
    refs = parse_references(reference_text)
    history = []

    for i, raw_ref in enumerate(refs):
        print(f"\n📄 ITEM {i+1}/{len(refs)}: {raw_ref[:70]}...")
        search_target = extract_title(raw_ref)

        log_entry = {
            "ID": i + 1,
            "Original_Ref": raw_ref, "Found_Title": "N/A", "Found_Year": "N/A", "Match_Score": 0.0,
            "Match_Reason": "Failed", "DOI": "N/A", "Candidate_URLs": [], "Tried_URL_Logs": [],
            "Final_Source": "None", "Status": "Failed",
            "Saved_Filename": "N/A", "UMich_Proxy_Link": "", "Direct_DOI_Link": "",
            "Timestamp": datetime.now().strftime('%Y-%m-%d %H:%M')
        }

        try:
            doi = extract_doi_regex(raw_ref)
            found_title = "Unknown"
            approved = False
            item_data = None

            if doi:
                heartbeat(f"Direct DOI Match Extracted: {doi}")
                log_entry["DOI"] = doi; log_entry["Match_Score"] = 1.0; log_entry["Match_Reason"] = "Direct DOI Regex Match"
                approved = True
                try:
                    res = cr.works(ids=doi)
                    item_data = res['message']
                    found_title = item_data.get('title', ['Unknown'])[0]
                    log_entry["Found_Title"] = found_title
                except: pass
            else:
                res = cr.works(query=raw_ref, limit=1)
                if res['message']['items']:
                    item_data = res['message']['items'][0]
                    doi = item_data.get('DOI', 'N/A')
                    found_title = item_data.get('title', ['Unknown'])[0]
                    log_entry["Found_Title"] = found_title; log_entry["DOI"] = doi

                    emb1 = vector_model.encode(search_target, convert_to_tensor=True)
                    emb2 = vector_model.encode(found_title, convert_to_tensor=True)
                    score = float(util.cos_sim(emb1, emb2))
                    log_entry["Match_Score"] = round(score, 2)
                    heartbeat(f"Semantic Match: {score:.2f} | Title: {found_title[:45]}...")

                    if score > 0.55:
                        approved = True; log_entry["Match_Reason"] = "Title Similarity High"
                    else:
                        try:
                            pub_year = str(item_data.get('issued', {}).get('date-parts', [[0]])[0][0])
                            authors = [a.get('family', '').lower() for a in item_data.get('author', [])]
                            log_entry["Found_Year"] = pub_year
                            if pub_year != '0' and pub_year in raw_ref:
                                if any(fuzzy_match(auth, raw_ref.lower()) > 0.6 for auth in authors) or \
                                   any(len(auth) > 2 and auth in raw_ref.lower() for auth in authors):
                                    approved = True; log_entry["Match_Reason"] = "ACS Format (Fuzzy Author/Year)"
                                    heartbeat("✅ Author & Year matched. Approving!")
                        except: pass

            if approved and doi != 'N/A':
                # Populate Manual Backup Links immediately
                log_entry["Direct_DOI_Link"] = f"https://doi.org/{doi}"
                log_entry["UMich_Proxy_Link"] = generate_umich_proxy(doi)

                candidate_urls = []
                heartbeat("Querying Databases (Unpaywall, SemScholar, OpenAlex, PMC, OA, Sci-Hub)...")

                candidate_urls.extend(fetch_unpaywall(doi))
                candidate_urls.extend(fetch_openalex(doi))
                candidate_urls.extend(fetch_semantic_scholar(doi))
                candidate_urls.extend(fetch_europe_pmc(doi))
                candidate_urls.extend(fetch_ncbi_pmc(doi))
                candidate_urls.extend(fetch_oaworks(doi))
                candidate_urls.extend(fetch_core_ac_uk(doi))
                candidate_urls.extend(fetch_scihub(doi))

                if item_data:
                     for link in item_data.get('link', []):
                         if link.get('content-type') == 'application/pdf' and link.get('intended-application') == 'text-mining':
                             candidate_urls.insert(0, ("Crossref Text-Mining", rewrite_direct_pdf_url(link.get('URL'))))
                         elif link.get('content-type') == 'application/pdf':
                             candidate_urls.append(("Crossref Direct", rewrite_direct_pdf_url(link.get('URL'))))

                unique_candidates = []
                seen_urls = set()
                for src, curl in candidate_urls:
                    if curl and curl not in seen_urls:
                        unique_candidates.append((src, curl))
                        seen_urls.add(curl)

                log_entry["Candidate_URLs"] = [u for s, u in unique_candidates]

                if unique_candidates:
                    file_name = clean_filename(found_title) if found_title != "Unknown" else f"{doi.replace('/','_')}.pdf"
                    save_path = os.path.join(DOWNLOAD_DIR, file_name)
                    dl_success = False

                    heartbeat(f"Found {len(unique_candidates)} potential download links.")

                    for source_name, url in unique_candidates:
                        heartbeat(f"Attempting Download via {source_name}...")
                        dl_result = verify_and_save_pdf(url, save_path)
                        log_entry["Tried_URL_Logs"].append(f"{source_name}: {dl_result}")

                        if dl_result == "Success":
                            log_entry["Status"] = "✅ Auto-Downloaded"
                            log_entry["Final_Source"] = source_name
                            log_entry["Saved_Filename"] = file_name
                            heartbeat(f"✅ SAVED CLEAN PDF: {file_name}")
                            dl_success = True
                            break
                        else:
                            heartbeat(f"❌ Link Failed ({dl_result}). Trying next...")
                            staggered_wait("Anti-bot pause", max_s=1.5)

                    if not dl_success:
                        log_entry["Status"] = "⚠️ Manual Download Required (WAF Block)"
                        heartbeat(f"❌ Automation Blocked. Manual Link: {log_entry['UMich_Proxy_Link']}")
                else:
                    log_entry["Status"] = "⚠️ Manual Download Required (No OA Links)"
                    heartbeat(f"❌ Paper is Paywalled. Manual Link: {log_entry['UMich_Proxy_Link']}")
            elif not approved:
                log_entry["Status"] = "❌ Match Failed"
                heartbeat("⚠️ Match score too low and Author/Year check failed.")
            else:
                log_entry["Status"] = "❌ Not Found"
                heartbeat("❌ Could not resolve this citation.")

        except Exception as e:
            log_entry["Status"] = f"Crash: {str(e)[:20]}"
            heartbeat(f"💥 Error: {log_entry['Status']}")

        log_entry["Candidate_URLs"] = " | ".join(log_entry["Candidate_URLs"])
        log_entry["Tried_URL_Logs"] = " | ".join(log_entry["Tried_URL_Logs"])
        history.append(log_entry)
        pd.DataFrame(history).to_csv(LOG_FILE, index=False)
        staggered_wait("Inter-paper Cooldown", 3)

    print("\n" + "="*50)
    print(f"Done! Check your Drive folder: {DOWNLOAD_DIR}")
    print(f"Verbose Debug Log Saved to: {LOG_FILE}")
    print("="*50)

    # ==========================================
    # 7. GENERATE THE MANUAL FALLBACK DASHBOARD
    # ==========================================
    print("\n📊 Generating Interactive Fulfillment Dashboard...")
    df = pd.DataFrame(history)

    # Filter columns for the dashboard
    display_cols = ["ID", "Original_Ref", "Status", "Saved_Filename", "UMich_Proxy_Link", "Direct_DOI_Link"]
    df_display = df[display_cols].copy()

    def make_umich_btn(val):
        if not val: return ""
        return f'<a href="{val}" target="_blank" style="background-color: #00274C; color: #FFCB05; padding: 4px 8px; text-decoration: none; border-radius: 4px; font-weight: bold; font-size: 12px; white-space: nowrap;">U-M Proxy</a>'

    def make_doi_btn(val):
        if not val: return ""
        return f'<a href="{val}" target="_blank" style="background-color: #555; color: white; padding: 4px 8px; text-decoration: none; border-radius: 4px; font-weight: bold; font-size: 12px; white-space: nowrap;">DOI Link</a>'

    styled_df = df_display.style.format({
        'UMich_Proxy_Link': make_umich_btn,
        'Direct_DOI_Link': make_doi_btn
    }).set_properties(subset=['Original_Ref'], **{'width': '300px', 'white-space': 'normal'}) \
      .set_properties(**{'text-align': 'left', 'vertical-align': 'top'}) \
      .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '14px'), ('background-color', '#f4f4f4')]},
        {'selector': 'td', 'props': [('font-size', '13px'), ('padding', '8px'), ('border-bottom', '1px solid #ddd')]}
    ]).hide(axis="index")

    display(HTML(styled_df.to_html()))

# Assuming target_references is defined in your previous cell
try: run_hybrid_workflow(target_references)
except NameError: print("⚠️ Error: 'target_references' variable not found.")

📦 Installing required packages (curl_cffi, bs4, habanero, sentence-transformers)...
Mounted at /content/drive
🚀 Initializing APEX Engine (Full Automation + Manual Fallback Dashboard)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Tools Ready. Saving PDFs to: /content/drive/MyDrive/uMich/Research/Cernak Lab - Trees/PhD-ish/Data/3 24/SystematicReview_Project/PDFs

📄 ITEM 1/26: (1) Mahjour, B.; Shen, Y.; Cernak, T. Ultrahigh-throughput experimenta...
  [17:50:34] Semantic Match: 0.92 | Title: Ultrahigh-Throughput Experimentation for Info...
  [17:50:34] Querying Databases (Unpaywall, SemScholar, OpenAlex, PMC, OA, Sci-Hub)...
  [17:50:39] ❌ Paper is Paywalled. Manual Link: https://proxy.lib.umich.edu/login?url=https://doi.org/10.1021/acs.accounts.1c00119
  [17:50:39] Inter-paper Cooldown: 1.3s...

📄 ITEM 2/26: (2) Gesmundo, N. J.; Sauvagnat, B.; Curran, P. J.; Richards, M. P.; An...
  [17:50:42] Semantic Match: 0.85 | Title: Nanoscale synthesis and affinity ranking...
  [17:50:42] Querying Databases (Unpaywall, SemScholar, OpenAlex, PMC, OA, Sci-Hub)...
  [17:50:45] Found 1 potential download links.
  [17:50:45] Attempting Download via Crossref Text-Mining...
  [17:50:45] Hit 404 Dead Link. Asking the Wayback Ma

ID,Original_Ref,Status,Saved_Filename,UMich_Proxy_Link,Direct_DOI_Link
1,"(1) Mahjour, B.; Shen, Y.; Cernak, T. Ultrahigh-throughput experimentation for information-rich chemical synthesis. Acc. Chem. Res. 2021, 54, 2337-2346.",⚠️ Manual Download Required (No OA Links),N/A,U-M Proxy,DOI Link
2,"(2) Gesmundo, N. J.; Sauvagnat, B.; Curran, P. J.; Richards, M. P.; Andrews, C. L.; Dandliker, P. J.; Cernak, T. Nanoscale synthesis and affinity ranking. Nature 2018, 557, 228-232.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
3,"(3) Uehling, M. R.; King, R. P.; Krska, S. W.; Cernak, T.; Buchwald, S. L. Pharmaceutical diversification via palladium oxidative addition complexes. Science 2019, 363, 405.",⚠️ Manual Download Required (No OA Links),N/A,U-M Proxy,DOI Link
4,"(4) Lin, S.; Dikler, S.; Blincoe, W. D.; Ferguson, R. D.; Sheridan, R. P.; Peng, Z.; Conway, D. V.; Zawatzky, K.; Wang, H.; Cernak, T.; Davies, I. W.; DiRocco, D. A.; Sheng, H.; Welch, C. J.; Dreher, S. D. Mapping the dark space of chemical reactions with extended nanomole synthesis and MALDI-TOF MS. Science 2018, 361, No. eaar6236.",⚠️ Manual Download Required (No OA Links),N/A,U-M Proxy,DOI Link
5,"(5) Buitrago Santanilla, A.; Regalado, E. L.; Pereira, T.; Shevlin, M.; Bateman, K.; Campeau, L.-C.; Schneeweis, J.; Berritt, S.; Shi, Z.-C.; Nantermet, P.; Liu, Y.; Helmy, R.; Welch, C. J.; Vachal, P.; Davies, I. W.; Cernak, T.; Dreher, S. D. Nanomole-scale high-throughput chemistry for the synthesis of complex molecules. Science 2015, 347, 49.",⚠️ Manual Download Required (No OA Links),N/A,U-M Proxy,DOI Link
6,"(6) Shen, Y.; Borowski, J. E.; Hardy, M. A.; Sarpong, R.; Doyle, A. G.; Cernak, T. Automation and computer-assisted planning for chemical synthesis. Nat. Rev. Methods Primers 2021, 1, 23.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
7,"(7) Torres, J. A. G.; Lau, S. H.; Anchuri, P.; Stevens, J. M.; Tabora, J. E.; Li, J.; Borovika, A.; Adams, R. P.; Doyle, A. G. A Multi-Objective Active Learning Platform and Web App for Reaction Optimization. J. Am. Chem. Soc. 2022, 144, 19999-20007.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
8,"(8) Shields, B. J.; Stevens, J.; Li, J.; Parasram, M.; Damani, F.; Alvarado, J. I. M.; Janey, J. M.; Adams, R. P.; Doyle, A. G. Bayesian reaction optimization as a tool for chemical synthesis. Nature 2021, 590, 89-96.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
9,"(9) Williams, W. L.; Zeng, L.; Gensch, T.; Sigman, M. S.; Doyle, A. G.; Anslyn, E. V. The evolution of data-driven modeling in organic chemistry. ACS Cent. Sci. 2021, 7, 1622-1637.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
10,"(10) Gao, W.; Raghavan, P.; Coley, C. W. Autonomous platforms for data-driven organic synthesis. Nat. Commun. 2022, 13, 1075.",⚠️ Manual Download Required (WAF Block),N/A,U-M Proxy,DOI Link
